# Core GIS continuation lab
Explore line simplification, polygon offsetting, proximity graphs and shortest paths. These are small, transparent examples designed to make the algorithmic choices visible.

All distances are planar coordinate units. Choose a projected CRS and appropriate units before applying the methods to real geographic data.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
for folder in ["line-simplification", "polygon-offsetting", "proximity-graphs", "shortest-paths"]:
    sys.path.insert(0, str(ROOT / "implementations" / "python" / folder))

import matplotlib.pyplot as plt
from shapely.geometry import Polygon
from simplification import douglas_peucker, visvalingam_whyatt
from offsetting import offset_polygon
from proximity_graphs import gabriel_graph, relative_neighborhood_graph, euclidean_mst
from shortest_paths import dijkstra, astar


## 1. Line simplification

Douglas–Peucker uses maximum deviation from a replacement segment. Visvalingam–Whyatt removes vertices in increasing effective-triangle-area order. Their tolerances have different units and should not be compared numerically.

In [ ]:
line = [(0, 0), (1, .25), (2, -.1), (3, .7), (4, 0), (5, .15), (6, 0)]
dp = douglas_peucker(line, tolerance=.35)
vw = visvalingam_whyatt(line, area_threshold=.3)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(*zip(*line), "o-", label=f"Original ({len(line)} vertices)")
ax.plot(*zip(*dp), "o--", label=f"Douglas–Peucker ({len(dp)})")
ax.plot(*zip(*vw), "s-.", label=f"Visvalingam–Whyatt ({len(vw)})")
ax.set_aspect("equal", adjustable="datalim"); ax.legend(); ax.grid(True)
plt.show()


## 2. Polygon offsetting

Positive offsets expand the polygon; negative offsets contract it. Join style changes the corners. The example uses Shapely/GEOS, not Clipper2.

In [ ]:
parcel = Polygon([(0, 0), (8, 0), (8, 5), (5, 5), (5, 2), (0, 2)])
expanded = offset_polygon(parcel, 0.7, join_style="round")
contracted = offset_polygon(parcel, -0.4, join_style="bevel")
fig, ax = plt.subplots(figsize=(7, 5))
for geom, label in [(parcel, "Original"), (expanded, "Expanded"), (contracted, "Contracted")]:
    if not geom.is_empty:
        x, y = geom.exterior.xy
        ax.plot(x, y, label=f"{label}: area={geom.area:.2f}")
ax.set_aspect("equal"); ax.legend(); ax.grid(True); plt.show()


## 3. Proximity graphs

Gabriel edges use an empty diameter disk; relative-neighbourhood edges use an empty lune; the Euclidean MST minimises total tree length. This lab uses direct small-input baselines.

In [ ]:
points = [(0, 0), (2, 0), (3, 1), (2, 3), (0, 2), (1, 1)]
graphs = [("Gabriel", gabriel_graph(points)), ("RNG", relative_neighborhood_graph(points)), ("Euclidean MST", [(i, j) for i, j, _ in euclidean_mst(points)])]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, edges) in zip(axes, graphs):
    ax.scatter(*zip(*points), zorder=3)
    for i, j in edges:
        ax.plot([points[i][0], points[j][0]], [points[i][1], points[j][1]], "-", alpha=.75)
    ax.set_title(f"{name}: {len(edges)} edges"); ax.set_aspect("equal"); ax.grid(True)
plt.tight_layout(); plt.show()


## 4. Shortest paths

Dijkstra minimises accumulated non-negative edge cost. A* changes queue priority using a heuristic lower bound. The heuristic must be expressed in the same units as the edge cost and must not overestimate remaining cost if optimality is required.

In [ ]:
graph = {
    "A": [("B", 4), ("C", 1)],
    "B": [("D", 2)],
    "C": [("B", 1), ("D", 7)],
    "D": []
}
coords = {"A": (0, 0), "B": (2, 0), "C": (1, 0), "D": (3, 0)}
heuristic = lambda node, goal: abs(coords[node][0] - coords[goal][0])
print("Dijkstra:", dijkstra(graph, "A", "D"))
print("A*:", astar(graph, "A", "D", heuristic))


## 5. Point location: a different problem from point-in-polygon

A point-in-polygon test classifies a point against one ring. A planar point-location structure returns the face of a subdivision containing the query. Slab decompositions use x-event coordinates and y-ordered edges inside each open slab. Trapezoidal maps use a decomposition plus a search DAG. Boundary points, vertical edges, shared vertices and the unbounded face need explicit rules.

The repository has a research dossier for these structures, but does not yet claim a runnable trapezoidal-map implementation. That distinction is intentional: a brute-force polygon predicate is not a point-location data structure.

## Review questions

1. Why are Douglas–Peucker and Visvalingam thresholds dimensionally different?
2. What happens to a narrow corridor under a negative offset?
3. Why can the Gabriel graph include a diagonal when another point lies exactly on the diameter circle?
4. What makes an A* heuristic admissible for a travel-time graph?
5. Which additional data structure is needed to answer planar face-location queries efficiently?
